In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


install libraries

In [2]:
!pip install opencv-python-headless==4.5.5.64 tensorboardx==2.4 "torch>=2.0.0" "torchvision>=0.15.0" "protobuf~=3.20.0"

# Evaluation Metrics and Definitions

In [3]:
from tensorboardX import SummaryWriter
class Visualization:
    def __init__(self):
        self.writer = ''

    def create_summary(self, model_type='U_Net'):
        self.writer = SummaryWriter(comment='-' +model_type)

    def add_scalar(self, epoch, value, params='loss'):
        self.writer.add_scalar(params, value, global_step=epoch)

    def add_iamge(self, tag, img_tensor):
        self.writer.add_iamge(tag, img_tensor)

    def add_graph(self, model):
        self.writer.add_graph(model)

    def close_summary(self):
        self.writer.close()

In [4]:
import numpy as np


class Evaluator(object):
    def __init__(self, num_class):
        self.num_class = num_class
        self.confusion_matrix = np.zeros((self.num_class,)*2)

    def get_tp_fp_tn_fn(self):
        tp = np.diag(self.confusion_matrix)
        fp = self.confusion_matrix.sum(axis=0) - np.diag(self.confusion_matrix)
        fn = self.confusion_matrix.sum(axis=1) - np.diag(self.confusion_matrix)
        tn = np.diag(self.confusion_matrix).sum() - np.diag(self.confusion_matrix)
        return tp, fp, tn, fn

    def Precision(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        precision = tp / (tp + fp)
        return precision

    def Recall(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        recall = tp / (tp + fn)
        return recall

    def F1(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        Precision = tp / (tp + fp )
        Recall = tp / (tp + fn)
        F1 = (2.0 * Precision * Recall) / (Precision + Recall)
        return F1

    def OA(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        OA = (tp+tn) / (tp + fp+tn+fn)
        return OA

    def Kappa(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        PRE = ((tp + fp) * (tp + fn) + (tn + fn) * (fp + tn)) / ((tp + fp + tn + fn) * (tp + fp + tn + fn))
        OA = (tp + tn) / (tp + fp + tn + fn)
        Kappa = (OA - PRE) / (1 - PRE)
        return Kappa

    def Pixel_Accuracy(self):
        Acc = np.diag(self.confusion_matrix).sum() / self.confusion_matrix.sum()
        return Acc

    def Pixel_Accuracy_Class(self):
        Acc = np.diag(self.confusion_matrix) / self.confusion_matrix.sum(axis=1)
        Acc = np.nanmean(Acc)
        return Acc

    def Mean_Intersection_over_Union(self):
        MIoU = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))
        MIoU = np.nanmean(MIoU)
        return MIoU

    def Intersection_over_Union(self):
        IoU = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))
        return IoU

    def Frequency_Weighted_Intersection_over_Union(self):
        freq = np.sum(self.confusion_matrix, axis=1) / np.sum(self.confusion_matrix)
        iu = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))

        FWIoU = (freq[freq > 0] * iu[freq > 0]).sum()
        return FWIoU

    def _generate_matrix(self, gt_image, pre_image):
        mask = (gt_image >= 0) & (gt_image < self.num_class)
        label = self.num_class * gt_image[mask].astype('int') + pre_image[mask]
        count = np.bincount(label, minlength=self.num_class**2)
        confusion_matrix = count.reshape(self.num_class, self.num_class)
        return confusion_matrix

    def add_batch(self, gt_image, pre_image):
        assert gt_image.shape == pre_image.shape
        self.confusion_matrix += self._generate_matrix(gt_image, pre_image)

    def reset(self):
        self.confusion_matrix = np.zeros((self.num_class,) * 2)

# Data Loader

In [5]:
import os
from PIL import Image
import torch.utils.data as data
import torchvision.transforms as transforms
import numpy as np
import random
from PIL import ImageEnhance
from PIL import ImageFilter

def safe_load_and_scale(path):
    img = Image.open(path)
    arr = np.array(img)

    # Check if the image array is 16-bit
    if arr.dtype == np.uint16 or arr.dtype == np.int16:
        # Calculate the 2nd and 98th percentiles to clip extreme bright/dark outliers
        p2, p98 = np.percentile(arr, (2, 98))

        # Clip the array to this range
        arr = np.clip(arr, p2, p98)

        # Min-max scale the clipped values to 0-255
        arr = (arr - p2) / (p98 - p2 + 1e-8) * 255.0
        arr = arr.astype(np.uint8)

        img = Image.fromarray(arr)

    return img.convert('RGB')

def cv_random_flip(img_A, img_B, label):
    # left right flip
    flip_flag = random.randint(0, 1)
    if flip_flag == 1:
        img_A = img_A.transpose(Image.FLIP_LEFT_RIGHT)
        img_B = img_B.transpose(Image.FLIP_LEFT_RIGHT)
        label = label.transpose(Image.FLIP_LEFT_RIGHT)
    return img_A, img_B, label

def cv_random_vflip(img_A, img_B, label):
    if random.randint(0, 1) == 1:
        img_A = img_A.transpose(Image.FLIP_TOP_BOTTOM)
        img_B = img_B.transpose(Image.FLIP_TOP_BOTTOM)
        label = label.transpose(Image.FLIP_TOP_BOTTOM)
    return img_A, img_B, label

def randomBlur(image_A, image_B):
    if random.random() > 0.5:
        radius = random.uniform(0.5, 1.5)
        image_A = image_A.filter(ImageFilter.GaussianBlur(radius=radius))
        image_B = image_B.filter(ImageFilter.GaussianBlur(radius=radius))
    return image_A, image_B

def randomCrop_Mosaic(image_A, image_B, label, crop_win_width, crop_win_height):
    image_width = image_A.size[0]
    image_height = image_A.size[1]
    random_region = (
        (image_width - crop_win_width) >> 1, (image_height - crop_win_height) >> 1, (image_width + crop_win_width) >> 1,
        (image_height + crop_win_height) >> 1)
    return image_A.crop(random_region), image_B.crop(random_region), label.crop(random_region)


def randomCrop(image_A, image_B, label):
    border = 30
    image_width = image_A.size[0]
    image_height = image_B.size[1]
    crop_win_width = np.random.randint(image_width - border, image_width)
    crop_win_height = np.random.randint(image_height - border, image_height)
    random_region = (
        (image_width - crop_win_width) >> 1, (image_height - crop_win_height) >> 1, (image_width + crop_win_width) >> 1,
        (image_height + crop_win_height) >> 1)
    return image_A.crop(random_region), image_B.crop(random_region), label.crop(random_region)


def randomRotation(image_A, image_B, label):
    mode = Image.BICUBIC
    if random.random() > 0.8:
        random_angle = np.random.randint(-15, 15)
        image_A = image_A.rotate(random_angle, mode)
        image_B = image_B.rotate(random_angle, mode)
        label = label.rotate(random_angle, mode)
    return image_A, image_B, label


def colorEnhance(image_A, image_B):
    bright = random.randint(5, 15) / 10.0
    contrast = random.randint(5, 15) / 10.0
    color = random.randint(0, 20) / 10.0
    sharp = random.randint(0, 30) / 10.0

    image_A = ImageEnhance.Brightness(image_A).enhance(bright)
    image_B = ImageEnhance.Brightness(image_B).enhance(bright)
    image_A = ImageEnhance.Contrast(image_A).enhance(contrast)
    image_B = ImageEnhance.Contrast(image_B).enhance(contrast)
    image_A = ImageEnhance.Color(image_A).enhance(color)
    image_B = ImageEnhance.Color(image_B).enhance(color)
    image_A = ImageEnhance.Sharpness(image_A).enhance(sharp)
    image_B = ImageEnhance.Sharpness(image_B).enhance(sharp)
    return image_A, image_B


def randomGaussian(image, mean=0.1, sigma=0.35):
    def gaussianNoisy(im, mean=mean, sigma=sigma):
        for _i in range(len(im)):
            im[_i] += random.gauss(mean, sigma)
        return im

    img = np.asarray(image)
    width, height = img.shape
    img = gaussianNoisy(img[:].flatten(), mean, sigma)
    img = img.reshape([width, height])
    return Image.fromarray(np.uint8(img))


def randomPeper(img):
    img = np.array(img)
    noiseNum = int(0.0015 * img.shape[0] * img.shape[1])
    for i in range(noiseNum):

        randX = random.randint(0, img.shape[0] - 1)

        randY = random.randint(0, img.shape[1] - 1)

        if random.randint(0, 1) == 0:

            img[randX, randY] = 0

        else:

            img[randX, randY] = 255
    return Image.fromarray(img)


# dataset for training
class ChangeDataset(data.Dataset):
    def __init__(self, root, trainsize, mosaic_ratio=0.75):
        self.trainsize = trainsize
        # get filenames
        self.image_root_A =  root + '/A/'
        self.image_root_B =  root + '/B/'
        self.gt_root = root + '/label/'
        self.mosaic_ratio = mosaic_ratio
        self.images_A = [self.image_root_A + f for f in os.listdir(self.image_root_A) if f.endswith('.jpg') or f.endswith('.png')]
        self.images_B = [self.image_root_B + f for f in os.listdir(self.image_root_B) if f.endswith('.jpg') or f.endswith('.png')]
        self.gts = [self.gt_root + f for f in os.listdir(self.gt_root) if f.endswith('.jpg')
                    or f.endswith('.png')]
        self.images_A = sorted(self.images_A)
        self.images_B = sorted(self.images_B)
        self.gts = sorted(self.gts)
        self.filter_files()
        # transforms
        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])
        # get size of dataset
        self.size = len(self.images_A)

    def __getitem__(self, index):
        # read imgs/gts/grads/depths
        p_ratio = random.random()
        if p_ratio > self.mosaic_ratio:
            image_A, image_B, gt = self.load_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = cv_random_vflip(image_A, image_B, gt)
            image_A, image_B, gt = randomCrop(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A, image_B = colorEnhance(image_A, image_B)
            image_A, image_B = randomBlur(image_A, image_B)
        else:
            image_A, image_B, gt = self.load_mosaic_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A,image_B = colorEnhance(image_A,image_B)

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()
        return image_A, image_B, gt


    def load_img_and_mask(self, index):
        A = Image.open(self.images_A[index]).convert('RGB')
        B = Image.open(self.images_B[index]).convert('RGB')
        mask = Image.open(self.gts[index]).convert('L')
        return A, B, mask

    def load_mosaic_img_and_mask(self, index):
       indexes = [index] + [random.randint(0, self.size - 1) for _ in range(3)]
       img_a_a, img_a_b, mask_a = self.load_img_and_mask(indexes[0])
       img_b_a, img_b_b, mask_b = self.load_img_and_mask(indexes[1])
       img_c_a, img_c_b, mask_c = self.load_img_and_mask(indexes[2])
       img_d_a, img_d_b, mask_d = self.load_img_and_mask(indexes[3])

       w = self.trainsize
       h = self.trainsize

       start_x = w // 4
       strat_y = h // 4
        # The coordinates of the splice center
       offset_x = random.randint(start_x, (w - start_x))
       offset_y = random.randint(strat_y, (h - strat_y))

       crop_size_a = (offset_x, offset_y)
       crop_size_b = (w - offset_x, offset_y)
       crop_size_c = (offset_x, h - offset_y)
       crop_size_d = (w - offset_x, h - offset_y)

       croped_a_a, croped_a_b, mask_crop_a = randomCrop_Mosaic(img_a_a.copy(), img_a_b.copy(), mask_a.copy(),crop_size_a[0], crop_size_a[1])
       croped_b_a, croped_b_b, mask_crop_b = randomCrop_Mosaic(img_b_a.copy(), img_b_b.copy(), mask_b.copy(),crop_size_b[0], crop_size_b[1])
       croped_c_a, croped_c_b, mask_crop_c = randomCrop_Mosaic(img_c_a.copy(), img_c_b.copy(), mask_c.copy(),crop_size_c[0], crop_size_c[1])
       croped_d_a, croped_d_b, mask_crop_d = randomCrop_Mosaic(img_d_a.copy(), img_d_b.copy(), mask_d.copy(),crop_size_d[0], crop_size_d[1])

       croped_a_a, croped_a_b, mask_crop_a = np.array(croped_a_a), np.array(croped_a_b), np.array(mask_crop_a)
       croped_b_a, croped_b_b, mask_crop_b = np.array(croped_b_a), np.array(croped_b_b), np.array(mask_crop_b)
       croped_c_a, croped_c_b, mask_crop_c = np.array(croped_c_a), np.array(croped_c_b), np.array(mask_crop_c)
       croped_d_a, croped_d_b, mask_crop_d = np.array(croped_d_a), np.array(croped_d_b), np.array(mask_crop_d)

       top_a = np.concatenate((croped_a_a, croped_b_a), axis=1)
       bottom_a = np.concatenate((croped_c_a, croped_d_a), axis=1)
       img_a = np.concatenate((top_a, bottom_a), axis=0)

       top_b = np.concatenate((croped_a_b, croped_b_b), axis=1)
       bottom_b = np.concatenate((croped_c_b, croped_d_b), axis=1)
       img_b = np.concatenate((top_b, bottom_b), axis=0)

       top_mask = np.concatenate((mask_crop_a, mask_crop_b), axis=1)
       bottom_mask = np.concatenate((mask_crop_c, mask_crop_d), axis=1)
       mask = np.concatenate((top_mask, bottom_mask), axis=0)
       mask = np.ascontiguousarray(mask)


       img_a = np.ascontiguousarray(img_a)
       img_b = np.ascontiguousarray(img_b)

       img_a = Image.fromarray(img_a)
       img_b = Image.fromarray(img_b)
       mask = Image.fromarray(mask)

       return img_a, img_b, mask

    def filter_files(self):
        assert len(self.images_A) == len(self.gts)
        assert len(self.images_A) == len(self.images_B)
        images_A = []
        images_B = []
        gts = []
        edges = []
        for img_A_path, img_B_path, gt_path in zip(self.images_A, self.images_B, self.gts):
            img_A = Image.open(img_A_path)
            img_B = Image.open(img_B_path)
            gt = Image.open(gt_path)
            if img_A.size == img_B.size:
                if img_A.size == gt.size:
                    images_A.append(img_A_path)
                    images_B.append(img_B_path)
                    gts.append(gt_path)

        self.images_A = images_A
        self.images_B = images_B
        self.gts = gts


    def __len__(self):
        return self.size





class Test_ChangeDataset(data.Dataset):
    def __init__(self, root, trainsize):
        self.trainsize = trainsize

        image_root_A = os.path.join(root, '2019') + '/'
        image_root_B = os.path.join(root, '2023') + '/'
        gt_root = os.path.join(root, 'labels') + '/'

        # Look for TIFFs in the image folders
        self.images_A = [image_root_A + f for f in os.listdir(image_root_A) if f.endswith(('.tif', '.TIF'))]
        self.images_B = [image_root_B + f for f in os.listdir(image_root_B) if f.endswith(('.tif', '.TIF'))]
        # Look for PNGs in the labels folder
        self.gts = [gt_root + f for f in os.listdir(gt_root) if f.endswith('.png')]

        self.images_A = sorted(self.images_A)
        self.images_B = sorted(self.images_B)
        self.gts = sorted(self.gts)
        self.filter_files()

        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])
        self.size = len(self.images_A)

    def __getitem__(self, index):
        image_A = self.rgb_loader(self.images_A[index])
        image_B = self.rgb_loader(self.images_B[index])
        gt = self.binary_loader(self.gts[index])

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()

        # Strip the 4-character .tif extension instead of .png
        file_name = self.images_A[index].split('/')[-1][:-4]

        return image_A, image_B, gt, file_name

    def filter_files(self):
        assert len(self.images_A) == len(self.gts)
        assert len(self.images_A) == len(self.images_B)
        images_A = []
        images_B = []
        gts = []
        for img_A_path, img_B_path, gt_path in zip(self.images_A, self.images_B, self.gts):
            img_A = Image.open(img_A_path)
            img_B = Image.open(img_B_path)
            gt = Image.open(gt_path)
            if img_A.size == img_B.size:
                if img_A.size == gt.size:
                    images_A.append(img_A_path)
                    images_B.append(img_B_path)
                    gts.append(gt_path)
        self.images_A = images_A
        self.images_B = images_B
        self.gts = gts

    def rgb_loader(self, path):
        # Replaced standard open with 16-bit safe loader
        return safe_load_and_scale(path)

    def binary_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('L')

    def __len__(self):
        return self.size



def get_loader(root, batchsize, trainsize, num_workers=1, shuffle=True, pin_memory=True):

    dataset =ChangeDataset(root = root, trainsize= trainsize)
    data_loader = data.DataLoader(dataset=dataset,
                                  batch_size=batchsize,
                                  shuffle=shuffle,
                                  num_workers=num_workers,
                                  pin_memory=pin_memory)
    return data_loader


def get_test_loader(root, batchsize, trainsize, num_workers=1, shuffle=True, pin_memory=True):

    dataset =Test_ChangeDataset(root = root, trainsize= trainsize)
    data_loader = data.DataLoader(dataset=dataset,
                                  batch_size=batchsize,
                                  shuffle=shuffle,
                                  num_workers=num_workers,
                                  pin_memory=pin_memory)
    return data_loader



class SemiChangeDataset(data.Dataset):
    def __init__(self, root, trainsize, train_ratio=1, mosaic_ratio=0.25):
        self.trainsize = trainsize
        self.mosaic_ratio = mosaic_ratio

        self.labeled_A = os.path.join(root, 'Labelled', 'train', '2019')
        self.labeled_B = os.path.join(root, 'Labelled', 'train', '2023')
        self.labeled_gt = os.path.join(root, 'Labelled', 'train', 'labels')

        self.unlabeled_A = os.path.join(root, 'Unlabelled', '2019')
        self.unlabeled_B = os.path.join(root, 'Unlabelled', '2023')

        # Load Labelled files (TIF for images, PNG for masks)
        self.img_A_labeled = sorted([os.path.join(self.labeled_A, f) for f in os.listdir(self.labeled_A) if f.endswith(('.tif', '.TIF'))])
        self.img_B_labeled = sorted([os.path.join(self.labeled_B, f) for f in os.listdir(self.labeled_B) if f.endswith(('.tif', '.TIF'))])
        self.gts_labeled = sorted([os.path.join(self.labeled_gt, f) for f in os.listdir(self.labeled_gt) if f.endswith('.png')])

        # Load Unlabelled files (TIF only)
        self.img_A_unlabeled = sorted([os.path.join(self.unlabeled_A, f) for f in os.listdir(self.unlabeled_A) if f.endswith(('.tif', '.TIF'))])
        self.img_B_unlabeled = sorted([os.path.join(self.unlabeled_B, f) for f in os.listdir(self.unlabeled_B) if f.endswith(('.tif', '.TIF'))])

        self.images_A = self.img_A_labeled + self.img_A_unlabeled
        self.images_B = self.img_B_labeled + self.img_B_unlabeled

        self.gts = self.gts_labeled + [None] * len(self.img_A_unlabeled)
        self.gt_list = list(range(len(self.img_A_labeled)))

        print(f'Total Training Samples: {len(self.images_A)}')
        print(f'Labelled: {len(self.img_A_labeled)}, Unlabelled: {len(self.img_A_unlabeled)}')

        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])

        self.size = len(self.images_A)
        # Classify each labeled tile as positive or negative
        self.positive_indices = []
        self.negative_indices = []
        for idx in self.gt_list:
            mask = Image.open(self.gts[idx]).convert('L')
            if np.array(mask).max() > 0:
                self.positive_indices.append(idx)
            else:
                self.negative_indices.append(idx)

        print(f'Positive tiles: {len(self.positive_indices)}, Negative tiles: {len(self.negative_indices)}')

    def __getitem__(self, index):
        p_ratio = random.random()
        if p_ratio > self.mosaic_ratio:
            image_A, image_B, gt = self.load_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = cv_random_vflip(image_A, image_B, gt)
            image_A, image_B, gt = randomCrop(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A, image_B = colorEnhance(image_A, image_B)
            image_A, image_B = randomBlur(image_A, image_B)
        else:
            image_A, image_B, gt = self.load_mosaic_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A,image_B = colorEnhance(image_A,image_B)

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()
        with_label = index in self.gt_list
        return image_A, image_B, gt, with_label

    def load_img_and_mask(self, index):
        # Use the 16-bit safe loader for the satellite imagery
        A = safe_load_and_scale(self.images_A[index])
        B = safe_load_and_scale(self.images_B[index])

        if index in self.gt_list:
            mask = Image.open(self.gts[index]).convert('L')
        else:
            # Generate dummy mask for unlabelled data
            mask = Image.new('L', A.size, 0)

        return A, B, mask

    def load_mosaic_img_and_mask(self, index):
       if index in self.gt_list:
           pool = self.gt_list  # labeled index → only pull from labeled tiles
       else:
           pool = list(range(self.size))
       indexes = [index]
       indexes.append(random.choice(self.positive_indices))
       indexes += [random.choice(pool) for _ in range(2)]
       random.shuffle(indexes)
       img_a_a, img_a_b, mask_a = self.load_img_and_mask(indexes[0])
       img_b_a, img_b_b, mask_b = self.load_img_and_mask(indexes[1])
       img_c_a, img_c_b, mask_c = self.load_img_and_mask(indexes[2])
       img_d_a, img_d_b, mask_d = self.load_img_and_mask(indexes[3])

       w = self.trainsize
       h = self.trainsize

       start_x = w // 4
       strat_y = h // 4
       offset_x = random.randint(start_x, (w - start_x))
       offset_y = random.randint(strat_y, (h - strat_y))

       crop_size_a = (offset_x, offset_y)
       crop_size_b = (w - offset_x, offset_y)
       crop_size_c = (offset_x, h - offset_y)
       crop_size_d = (w - offset_x, h - offset_y)

       croped_a_a, croped_a_b, mask_crop_a = randomCrop_Mosaic(img_a_a.copy(), img_a_b.copy(), mask_a.copy(),crop_size_a[0], crop_size_a[1])
       croped_b_a, croped_b_b, mask_crop_b = randomCrop_Mosaic(img_b_a.copy(), img_b_b.copy(), mask_b.copy(),crop_size_b[0], crop_size_b[1])
       croped_c_a, croped_c_b, mask_crop_c = randomCrop_Mosaic(img_c_a.copy(), img_c_b.copy(), mask_c.copy(),crop_size_c[0], crop_size_c[1])
       croped_d_a, croped_d_b, mask_crop_d = randomCrop_Mosaic(img_d_a.copy(), img_d_b.copy(), mask_d.copy(),crop_size_d[0], crop_size_d[1])

       croped_a_a, croped_a_b, mask_crop_a = np.array(croped_a_a), np.array(croped_a_b), np.array(mask_crop_a)
       croped_b_a, croped_b_b, mask_crop_b = np.array(croped_b_a), np.array(croped_b_b), np.array(mask_crop_b)
       croped_c_a, croped_c_b, mask_crop_c = np.array(croped_c_a), np.array(croped_c_b), np.array(mask_crop_c)
       croped_d_a, croped_d_b, mask_crop_d = np.array(croped_d_a), np.array(croped_d_b), np.array(mask_crop_d)

       top_a = np.concatenate((croped_a_a, croped_b_a), axis=1)
       bottom_a = np.concatenate((croped_c_a, croped_d_a), axis=1)
       img_a = np.concatenate((top_a, bottom_a), axis=0)

       top_b = np.concatenate((croped_a_b, croped_b_b), axis=1)
       bottom_b = np.concatenate((croped_c_b, croped_d_b), axis=1)
       img_b = np.concatenate((top_b, bottom_b), axis=0)

       top_mask = np.concatenate((mask_crop_a, mask_crop_b), axis=1)
       bottom_mask = np.concatenate((mask_crop_c, mask_crop_d), axis=1)
       mask = np.concatenate((top_mask, bottom_mask), axis=0)
       mask = np.ascontiguousarray(mask)

       img_a = np.ascontiguousarray(img_a)
       img_b = np.ascontiguousarray(img_b)

       img_a = Image.fromarray(img_a)
       img_b = Image.fromarray(img_b)
       mask = Image.fromarray(mask)

       return img_a, img_b, mask

    def __len__(self):
        return self.size

def get_semiloader(root, batchsize, trainsize, train_ratio, num_workers=1, shuffle=True, pin_memory=True):
    from torch.utils.data import WeightedRandomSampler

    dataset = SemiChangeDataset(root=root, trainsize=trainsize, train_ratio=train_ratio)

    # positive tiles get higher weight, negative tiles get lower weight
    # Unlabeled tiles get same weight as negative (they're mostly negative)
    n_pos = len(dataset.positive_indices)
    n_neg = len(dataset.negative_indices)
    n_unlabeled = len(dataset.img_A_unlabeled)

    pos_set = set(dataset.positive_indices)
    neg_set = set(dataset.negative_indices)

    weights = []
    for idx in range(len(dataset)):
        if idx in pos_set:
            weights.append(1.0 / n_pos)
        elif idx in neg_set:
            weights.append(1.0 / n_neg)
        else:
            weights.append(1.0 / (n_neg + n_unlabeled))

    sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(dataset),
        replacement=True
    )

    data_loader = data.DataLoader(
        dataset=dataset,
        batch_size=batchsize,
        sampler=sampler,  # replaces shuffle
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    return data_loader

# Neural Network Architecture

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# Shared Attention Blocks
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        # Ensure ratio doesn't reduce channels below 1
        reduced = max(in_planes // ratio, 1)
        self.fc1 = nn.Conv2d(in_planes, reduced, 1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(reduced, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(combined))


# Decoder Block: upsample, skip, refine
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super(DecoderBlock, self).__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatches from odd input dimensions
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# Full U-Net Decoder
class UNetDecoder(nn.Module):
    def __init__(self, channel=64):
        super(UNetDecoder, self).__init__()

        # Reduce l5 to working channels
        self.conv5 = nn.Conv2d(512, channel * 4, 1)

        # Decoder path: l5 -> l4 -> l3 -> l2 -> l1
        self.dec4 = DecoderBlock(channel * 4, 512, channel * 4)  # fuse with l4
        self.dec3 = DecoderBlock(channel * 4, 256, channel * 2)  # fuse with l3
        self.dec2 = DecoderBlock(channel * 2, 128, channel)      # fuse with l2
        self.dec1 = DecoderBlock(channel, 64, channel)           # fuse with l1

        self.pred = nn.Sequential(
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.BatchNorm2d(channel),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel, 1, 1)
        )

    def forward(self, l1, l2, l3, l4, l5):
        x = self.conv5(l5)         # at 1/16 resolution
        x = self.dec4(x, l4)       # 1/8
        x = self.dec3(x, l3)       # 1/4
        x = self.dec2(x, l2)       # 1/2
        x = self.dec1(x, l1)       # 1/1
        return self.pred(x)


# Main Model
class SemiModel(nn.Module):
    def __init__(self, channel=64):
        super(SemiModel, self).__init__()

        vgg16_bn = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
        self.inc = vgg16_bn.features[:5]      # -> 64 channels, 1/1 res (after first pool, actually 1/2... see note)
        self.down1 = vgg16_bn.features[5:12]  # -> 128 channels, 1/2
        self.down2 = vgg16_bn.features[12:22] # -> 256 channels, 1/4
        self.down3 = vgg16_bn.features[22:32] # -> 512 channels, 1/8
        self.down4 = vgg16_bn.features[32:42] # -> 512 channels, 1/16

        # Attention at every difference level
        self.ca1 = ChannelAttention(64);  self.sa1 = SpatialAttention()
        self.ca2 = ChannelAttention(128); self.sa2 = SpatialAttention()
        self.ca3 = ChannelAttention(256); self.sa3 = SpatialAttention()
        self.ca4 = ChannelAttention(512); self.sa4 = SpatialAttention()
        self.ca5 = ChannelAttention(512); self.sa5 = SpatialAttention()

        self.decoder = UNetDecoder(channel)

    def forward(self, A, B):
        # Backbone (siamese)
        l1A = self.inc(A);   l1B = self.inc(B)
        l2A = self.down1(l1A); l2B = self.down1(l1B)
        l3A = self.down2(l2A); l3B = self.down2(l2B)
        l4A = self.down3(l3A); l4B = self.down3(l3B)
        l5A = self.down4(l4A); l5B = self.down4(l4B)

        # Absolute difference at every level
        d1 = torch.abs(l1A - l1B)
        d2 = torch.abs(l2A - l2B)
        d3 = torch.abs(l3A - l3B)
        d4 = torch.abs(l4A - l4B)
        d5 = torch.abs(l5A - l5B)

        # Apply attention
        d1 = d1 * self.ca1(d1) * self.sa1(d1)
        d2 = d2 * self.ca2(d2) * self.sa2(d2)
        d3 = d3 * self.ca3(d3) * self.sa3(d3)
        d4 = d4 * self.ca4(d4) * self.sa4(d4)
        d5 = d5 * self.ca5(d5) * self.sa5(d5)

        # Decoder uses all 5 levels
        pred = self.decoder(d1, d2, d3, d4, d5)

        # Final upsample to input size if needed
        if pred.shape[2:] != A.shape[2:]:
            pred = F.interpolate(pred, size=A.shape[2:], mode='bilinear', align_corners=True)

        attention_map = F.interpolate(d5.mean(dim=1, keepdim=True),
                                      size=A.shape[2:], mode='bilinear', align_corners=True)

        return attention_map, pred

# Model Training

Model Training Settings

In [7]:
import sys
sys.argv = [
    "notebook",
    "--model_name", "Dubai Building Footprint Change Detector 2",
    "--data_name", "DUBAI",
    "--lr", "5e-5",
    "--epoch", "100",
    "--gpu_id", "0",
    "--batchsize", "8",
    "--pretrained", "None",
    "--train_ratio", "0.2",
    "--use_burnin"
]

In [8]:
import time
import os
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import random
import time
start=time.time()

# Unfreeze specific layers of the pretrained VGG16 backbone during transfer learning
def unfreeze_encoder(model, ema_model, encoder_lr=1e-6):
    backbone_prefixes = ("inc", "down1", "down2", "down3", "down4")

    for name, param in model.named_parameters():
        if any(name.startswith(p) for p in backbone_prefixes):
            param.requires_grad = True

    encoder_params = [p for n, p in model.named_parameters()
                      if any(n.startswith(pfx) for pfx in backbone_prefixes) and p.requires_grad]
    decoder_params = [p for n, p in model.named_parameters()
                      if not any(n.startswith(pfx) for pfx in backbone_prefixes) and p.requires_grad]

    return encoder_params, decoder_params

# bad early predictions can poison/collapse the Semi-supervised model, thus ramp up the lambda over training epochs
def get_lambda(epoch):
    # ramp-up for semi-supervised weight
    if epoch < 5:
        return epoch / 5.0 * 0.2          # 0 → 0.2
    elif epoch < 20:
        return 0.2 + (epoch - 5) / 15.0 * 0.4  # 0.2 → 0.6
    else:
        return 0.6

# Controls the momentum for the EMA teacher model, makes it more stable after more training epochs.
def get_alpha(epoch, max_epoch):
    # EMA decay schedule
    if epoch < 10:
        return 0.95
    elif epoch < 30:
        return 0.97
    else:
        return 0.99

# Modifies standard Binary Cross-Entropy (BCE) by down-weighting easy/confident predictions and focusing on hard examples using the gamma parameter.

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # # Alpha handles class imbalance weight for POSITIVE (change) class
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        # alpha for positives, (1-alpha) for negatives
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# Measures  spatial overlap between  prediction and  ground truth using the Dice coefficient. effective for segmentation boundaries.
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)

        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice = (2.*intersection + self.smooth)/(inputs.sum() + targets.sum() + self.smooth)

        return 1 - dice

# Hybrid combination of pixel classification stability (Focal/BCE) with spatial overlap accuracy (Dice). We use a 50/50 split of Focal and Dice loss for supervised training.
class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, dice_weight=0.5):
        super(FocalDiceLoss, self).__init__()
        self.focal = FocalLoss(alpha, gamma)
        self.dice = DiceLoss()
        self.dice_weight = dice_weight

    def forward(self, inputs, targets):
        focal = self.focal(inputs, targets)
        dice = self.dice(inputs, targets)
        return (1 - self.dice_weight) * focal + self.dice_weight * dice

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.6, dice_weight=0.4):
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        dice_loss = self.dice(inputs, targets)
        # Combination: 60% BCE for stability, 40% Dice for overlap
        return (self.bce_weight * bce_loss) + (self.dice_weight * dice_loss)

# Lock down random numbers to ensure training results are reproducible
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


def update_ema_variables(model, ema_model, alpha):
    model_state = model.state_dict()
    model_ema_state = ema_model.state_dict()
    new_dict = {}
    for key in model_state:
        new_dict[key] = alpha * model_ema_state[key] + (1 - alpha) * model_state[key]
    ema_model.load_state_dict(new_dict)


def train1(train_loader, val_loader, Eva_train, Eva_train2, Eva_val, Eva_val2,
           data_name, save_path, net, ema_net, criterion, semicriterion,
           optimizer, use_ema, num_epoches, epoch):
    vis = Visualization()
    vis.create_summary(data_name)
    global best_iou
    epoch_loss = 0
    net.train(True)
    ema_net.train(True)

    length = 0
    st = time.time()
    loss_semi=torch.zeros(1)

    lambda_semi = get_lambda(epoch)
    alpha = get_alpha(epoch, num_epoches)

    with tqdm(total=len(train_loader), desc=f'Eps {epoch}/{num_epoches}', unit='img') as pbar:
        for i, (A, B, mask, with_label) in enumerate(train_loader):
            A = A.cuda()
            B = B.cuda()
            Y = mask.cuda()
            with_label = with_label.cuda()

            optimizer.zero_grad()

            if not use_ema:
                if with_label.any():
                    preds = net(A[with_label], B[with_label])
                    loss = criterion(preds[1], Y[with_label])
                    Y_used = Y[with_label]
                else:
                    continue
            else:
                preds = net(A, B)
                loss = 0.0
                if with_label.any():
                    loss = criterion(preds[1][with_label], Y[with_label])
                    Y_used = Y[with_label]
                else:
                    Y_used = Y  # not used for metrics if no labeled

                loss_semi = torch.zeros(1, device=A.device)

                if (~with_label).any():
                    with torch.no_grad():
                        z1 = A[~with_label]
                        z2 = B[~with_label]
                        pseudo_attn, pseudo_preds = ema_net(z1, z2)
                        pseudo_attn = torch.sigmoid(pseudo_attn)
                        pseudo_preds = torch.sigmoid(pseudo_preds)

                        # confidence thresholding
                        conf_thr = 0.7
                        mask_conf_attn = (pseudo_attn > conf_thr).float()
                        mask_conf_pred = (pseudo_preds > conf_thr).float()
                        pseudo_attn = pseudo_attn * mask_conf_attn
                        pseudo_preds = pseudo_preds * mask_conf_pred

                    loss_semi = semicriterion(preds[1][~with_label], pseudo_preds)

                    loss = loss + lambda_semi * loss_semi

                    # track metrics
                    mask_cpu = mask.cpu().numpy().astype(int)
                    pred_cpu = (preds[1].detach().cpu().numpy() > 0).astype(int)
                    with_label_cpu = with_label.cpu().numpy()
                    Eva_train2.add_batch(
                        mask_cpu[~with_label_cpu],
                        pred_cpu[~with_label_cpu]
                    )

            loss.backward()
            optimizer.step()

            with torch.no_grad():
                update_ema_variables(net, ema_net, alpha=alpha)

            epoch_loss += loss.item()

            if with_label.any():
                if use_ema:
                    output = torch.sigmoid(preds[1][with_label])
                else:
                    output = torch.sigmoid(preds[1])  # already filtered in burn-in
                output = (output >= 0.5).float()
                pred = output.data.cpu().numpy().astype(int)
                target = Y[with_label].cpu().numpy().astype(int)
                Eva_train.add_batch(target, pred)

            pbar.set_postfix(**{'LAll': loss.item(), 'LSemi': loss_semi.item()})
            pbar.update(1)
            length += 1



    IoU = Eva_train.Intersection_over_Union()[1]
    Pre = Eva_train.Precision()[1]
    Recall = Eva_train.Recall()[1]
    F1 = Eva_train.F1()[1]
    train_loss = epoch_loss / length

    vis.add_scalar(epoch, IoU, 'mIoU')
    vis.add_scalar(epoch, Pre, 'Precision')
    vis.add_scalar(epoch, Recall, 'Recall')
    vis.add_scalar(epoch, F1, 'F1')
    vis.add_scalar(epoch, train_loss, 'train_loss')

    print(
        'Epoch [%d/%d], Loss: %.4f,\n[Training]IoU: %.4f, Precision:%.4f, Recall: %.4f, F1: %.4f' % (
            epoch, num_epoches, \
            train_loss, \
            Eva_train.Intersection_over_Union()[1], Eva_train.Precision()[1], Eva_train.Recall()[1], Eva_train.F1()[1]))

    if use_ema is True:
        print(
            'Epoch [%d/%d],\n[Training]IoU: %.4f, Precision:%.4f, Recall: %.4f, F1: %.4f' % (
                epoch, num_epoches, \
                IoU, Pre, Recall, F1))
    print("Strat validing!")


    net.train(False)
    net.eval()
    ema_net.train(False)
    ema_net.eval()
    for i, (A, B, mask, filename) in enumerate(tqdm(val_loader)):
        with torch.no_grad():
            A = A.cuda()
            B = B.cuda()
            Y = mask.cuda()
            preds = net(A,B)[1]
            output = F.sigmoid(preds)
            output[output >= 0.5] = 1
            output[output < 0.5] = 0
            pred = output.data.cpu().numpy().astype(int)
            target = Y.cpu().numpy().astype(int)
            Eva_val.add_batch(target, pred)

            preds_ema = ema_net(A, B)[1]
            Eva_val2.add_batch(target, (preds_ema>0).cpu().numpy().astype(int))
            length += 1
            """
                It doesn't matter whether you store the parameters of net or ema_net here; it depends on which has higher precision.
            """
    IoU = Eva_val.Intersection_over_Union()
    Pre = Eva_val.Precision()
    Recall = Eva_val.Recall()
    F1 = Eva_val.F1()
    lr_scheduler.step(IoU[1])

    print('[Validation] IoU: %.4f, Precision:%.4f, Recall: %.4f, F1: %.4f' % (IoU[1], Pre[1], Recall[1], F1[1]))

    print('[Ema Validation] IoU: %.4f, Precision:%.4f, Recall: %.4f, F1: %.4f' % (Eva_val2.Intersection_over_Union()[1], Eva_val2.Precision()[1], Eva_val2.Recall()[1], Eva_val2.F1()[1]))
    new_iou = IoU[1]    #save teacher model？
    if new_iou >= best_iou:
        best_iou = new_iou
        best_epoch = epoch
        print('Best Model Iou :%.4f; F1 :%.4f; Best epoch : %d' % (IoU[1], F1[1], best_epoch))

        print('best_epoch', epoch)
        student_dir = save_path + '_train1_' + '_best_student_iou.pth'

        student_state = {'best_student_net ': net.state_dict(),
                 'optimizer ': optimizer.state_dict(),
                 ' epoch': epoch}

        torch.save(student_state, student_dir)
        torch.save(ema_net.state_dict(), save_path + '_train1_' + '_best_teacher_iou.pth')
    print('Best Model Iou :%.4f; F1 :%.4f' % (best_iou, F1[1]))
    vis.close_summary()

if __name__ == '__main__':
    seed_everything(42)
    import argparse

    parser = argparse.ArgumentParser()

    parser.add_argument('--resume', type=str, default=None, help='path to checkpoint to resume from (e.g. last.pth)')

    parser.add_argument('--pretrained', type=str, default='/content/drive/MyDrive/MBRSC_Project/C2F-SemiCD/savesLEVIR/C2FNet_FullSup_LEVIR_train1__best_student_iou.pth', help='path to LEVIR pre-trained weights')
    parser.add_argument('--use_burnin', action='store_true', help='Use supervised burn-in for first x epochs')
    # CHANGE DEFAULT LR FOR FINE-TUNING (From 5e-4 to 5e-5)
    parser.add_argument('--lr', type=float, default=5e-5, help='learning rate')
    parser.add_argument('--epoch', type=int, default=100, help='epoch number')
    parser.add_argument('--batchsize', type=int, default=16, help='training batch size')
    parser.add_argument('--trainsize', type=int, default=256, help='training dataset size')
    parser.add_argument('--train_ratio', type=float, default=1, help='Proportion of the labeled images')
    parser.add_argument('--clip', type=float, default=0.5, help='gradient clipping margin')
    parser.add_argument('--decay_rate', type=float, default=0.1, help='decay rate of learning rate')
    parser.add_argument('--decay_epoch', type=int, default=50, help='every n epochs decay learning rate')
    parser.add_argument('--gpu_id', type=str, default='0,1', help='train use gpu')
    parser.add_argument('--data_name', type=str, default='WHU',
                        help='the test rgb images root')
    parser.add_argument('--model_name', type=str, default='SemiModel_noema04',
                        help='the test rgb images root')
    # parser.add_argument('--save_path', type=str, default='./output/C2F-SemiCD/WHU-5/')  # Semi-supervised model save path
    parser.add_argument('--save_path', type=str, default='./output/C2FNet/WHU/')  # Fully supervised model save path

    opt = parser.parse_args()
    print('labeled ration=1,The coefficients of the semi-supervised loss function are now::0.2!')

    # set the device for training
    if opt.gpu_id == '0':
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        print('USE GPU 0')
    elif opt.gpu_id == '1':
        os.environ["CUDA_VISIBLE_DEVICES"] = "1"
        print('USE GPU 1')
    if opt.gpu_id == '2':
        os.environ["CUDA_VISIBLE_DEVICES"] = "2"
        print('USE GPU 2')
    if opt.gpu_id == '3':
        os.environ["CUDA_VISIBLE_DEVICES"] = "3"
        print('USE GPU 3')

    print("1. Setting up paths...")
    opt.save_path = opt.save_path + opt.data_name + '/' + opt.model_name

    print("1. Setting up paths...")
    opt.save_path = opt.save_path + opt.data_name + '/' + opt.model_name

    # CUSTOM DATASET ROUTE HERE:
    if opt.data_name == 'DUBAI':
        opt.train_root = '/content/drive/MyDrive/MBRSC_Project/Dubai Satellite Dataset - Training splits/Dubai Satellite Dataset - Training splits by category - expanded - 3'
        opt.val_root = '/content/drive/MyDrive/MBRSC_Project/Dubai Satellite Dataset - Training splits/Dubai Satellite Dataset - Training splits by category - expanded - 3/Labelled/val'

    print("2. Initializing train dataloader...")
    train_loader = get_semiloader(opt.train_root, opt.batchsize, opt.trainsize,opt.train_ratio, num_workers=2, shuffle=True, pin_memory=True) # 12 workers for collab's A100 model
    print("3. Initializing val dataloader...")
    val_loader = get_test_loader(opt.val_root, opt.batchsize, opt.trainsize, num_workers=2, shuffle=False, pin_memory=True)

    print("4. Initializing Evaluators...")
    Eva_train = Evaluator(num_class = 2)
    Eva_train2 = Evaluator(num_class=2)
    Eva_val = Evaluator(num_class=2)
    Eva_val2 = Evaluator(num_class=2)

    print("5. Moving SemiModel to GPU...")
    model=SemiModel().cuda()
    print("6. Moving EMA Model to GPU...")
    ema_model = SemiModel().cuda()
    print("7. Successfully moved models to GPU!")
    for param in ema_model.parameters():
        param.detach_()

    criterion = FocalDiceLoss(alpha=0.75, gamma=2.0, dice_weight=0.5).cuda()
    semicriterion = FocalLoss(alpha=0.75, gamma=2.0).cuda()

    # optimizer = torch.optim.Adam(model.parameters(), opt.lr)
    #base_optimizer = torch.optim.AdamW(model.parameters(), lr=opt.lr, weight_decay=0.0025)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=opt.lr, weight_decay=0.0025)

    #lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',       # because we track IoU
    factor=0.5,       # reduce LR by half
    patience=3,       # wait 3 epochs without improvement
    min_lr=1e-6)


    save_path = opt.save_path
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    data_name = opt.data_name
    best_iou = 0.0

    print("Start train...")
    # args = parser.parse_args()
    # print('The current data is：',args.data_name)

    start_epoch = 1

    if opt.resume is not None and os.path.isfile(opt.resume):
        print(f"=> Loading checkpoint from {opt.resume}")
        checkpoint = torch.load(opt.resume, map_location='cuda')
        model.load_state_dict(checkpoint['net'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch'] + 1
        if 'ema_net' in checkpoint:
            ema_model.load_state_dict(checkpoint['ema_net'])
        print(f"=> Resumed from epoch {start_epoch}")

    # FINE-TUNING INITIALIZATION
    elif opt.pretrained is not None and os.path.isfile(opt.pretrained):
        print(f"=> Loading pre-trained LEVIR weights from {opt.pretrained}")
        checkpoint = torch.load(opt.pretrained, map_location='cuda')
        state_dict = checkpoint['best_student_net '] if 'best_student_net ' in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=False)
        ema_model.load_state_dict(state_dict, strict=False)
        print("=> Initialized Student and Teacher with LEVIR weights.")
    else:
        print("=> Training from scratch.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=opt.lr, weight_decay=0.0025)

    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    for epoch in range(start_epoch, opt.epoch + 1):
        for param_group in optimizer.param_groups:
            print(param_group['lr'])

        if epoch <= 10 and opt.use_burnin:
            use_ema = False
            print(f"--- Epoch {epoch}: Supervised Burn-in (EMA disabled) ---")
        else:
            use_ema = True
            print(f"--- Epoch {epoch}: Semi-Supervised Mean Teacher (EMA enabled) ---")

        Eva_train.reset()
        Eva_train2.reset()
        Eva_val.reset()
        Eva_val2.reset()

        train1(train_loader, val_loader, Eva_train, Eva_train2, Eva_val, Eva_val2,
              data_name, save_path, model, ema_model,
              criterion, semicriterion, optimizer, use_ema, opt.epoch, epoch)

        last_checkpoint = {
            'net': model.state_dict(),
            'ema_net': ema_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch
        }

        torch.save(
            last_checkpoint,
            os.path.join(save_path, 'last.pth')
        )

        print(f"=> Saved last checkpoint (epoch {epoch})")

end=time.time()
print('The training time for the program is:',end-start)

labeled ration=1,The coefficients of the semi-supervised loss function are now::0.2!
USE GPU 0
1. Setting up paths...
1. Setting up paths...
2. Initializing train dataloader...
Total Training Samples: 578
Labelled: 200, Unlabelled: 378
Positive tiles: 52, Negative tiles: 148
3. Initializing val dataloader...
4. Initializing Evaluators...
5. Moving SemiModel to GPU...
6. Moving EMA Model to GPU...
7. Successfully moved models to GPU!
Start train...
=> Training from scratch.
5e-05
--- Epoch 1: Supervised Burn-in (EMA disabled) ---


Eps 1/100: 100%|██████████| 73/73 [04:47<00:00,  3.94s/img, LAll=0.314, LSemi=0]


Epoch [1/100], Loss: 0.4156,
[Training]IoU: 0.2050, Precision:0.2268, Recall: 0.6805, F1: 0.3402
Strat validing!


100%|██████████| 8/8 [00:04<00:00,  1.90it/s]


[Validation] IoU: 0.3239, Precision:0.3561, Recall: 0.7819, F1: 0.4893
[Ema Validation] IoU: 0.0100, Precision:0.5607, Recall: 0.0101, F1: 0.0198
Best Model Iou :0.3239; F1 :0.4893; Best epoch : 1
best_epoch 1
Best Model Iou :0.3239; F1 :0.4893
=> Saved last checkpoint (epoch 1)
5e-05
--- Epoch 2: Supervised Burn-in (EMA disabled) ---


...

[Validation] IoU: 0.5608, Precision:0.7288, Recall: 0.7087, F1: 0.7186
[Ema Validation] IoU: 0.4558, Precision:0.6106, Recall: 0.6425, F1: 0.6261
Best Model Iou :0.5880; F1 :0.7186
=> Saved last checkpoint (epoch 99)
1e-06
--- Epoch 100: Semi-Supervised Mean Teacher (EMA enabled) ---


Eps 100/100: 100%|██████████| 73/73 [00:57<00:00,  1.28img/s, LAll=0.289, LSemi=0.0197]


Epoch [100/100], Loss: 0.1063,
[Training]IoU: 0.8313, Precision:0.8822, Recall: 0.9351, F1: 0.9079
Epoch [100/100],
[Training]IoU: 0.8313, Precision:0.8822, Recall: 0.9351, F1: 0.9079
Strat validing!


100%|██████████| 8/8 [00:04<00:00,  1.97it/s]


[Validation] IoU: 0.5024, Precision:0.8195, Recall: 0.5649, F1: 0.6688
[Ema Validation] IoU: 0.4173, Precision:0.7424, Recall: 0.4880, F1: 0.5889
Best Model Iou :0.5880; F1 :0.6688
=> Saved last checkpoint (epoch 100)
The training time for the program is: 6445.193475484848


# Model Inference

Find the optimal threshold to use during model inference

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# Load model checkpoint
model = SemiModel().cuda()
checkpoint = torch.load('./Fine-Tuned Dubai Building Footprint Change Detection Model Best Checkpoint/Focal Loss Expanded 3 lr=5e-5 blobfix archfix 100epochs NoPretraining_train1__best_student_iou.pth', map_location='cuda')
state_dict = checkpoint['best_student_net '] if 'best_student_net ' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

# Load data
val_loader = get_test_loader(
    '/content/drive/MyDrive/MBRSC_Project/Dubai Satellite Dataset - Training splits/Dubai Satellite Dataset - Training splits by category - expanded - 3/Labelled/test',
    batchsize=8, trainsize=256, num_workers=2, shuffle=False, pin_memory=True
)

# Collect all predictions and targets
all_probs = []
all_targets = []

with torch.no_grad():
    for A, B, mask, filename in val_loader:
        A, B = A.cuda(), B.cuda()
        preds = model(A, B)[1]
        probs = torch.sigmoid(preds).cpu().numpy()
        targets = mask.numpy().astype(int)
        all_probs.append(probs)
        all_targets.append(targets)

all_probs = np.concatenate(all_probs, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# Sweep thresholds for model inference
print(f"{'Threshold':>10} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 50)

best_thr, best_iou = 0.5, 0
for thr in np.arange(0.30, 0.75, 0.05):
    preds_bin = (all_probs >= thr).astype(int)

    tp = ((preds_bin == 1) & (all_targets == 1)).sum()
    fp = ((preds_bin == 1) & (all_targets == 0)).sum()
    fn = ((preds_bin == 0) & (all_targets == 1)).sum()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    marker = " <-- best" if iou > best_iou else ""
    print(f"{thr:>10.2f} {iou:>8.4f} {precision:>10.4f} {recall:>8.4f} {f1:>8.4f}{marker}")

    if iou > best_iou:
        best_iou = iou
        best_thr = thr

print(f"\nOptimal threshold: {best_thr:.2f}, IoU: {best_iou:.4f}")

In [ ]:
!pip install "numpy<2"

Run model inference on all dataset tiles, using time-tested augmentation

In [ ]:
import os, time, sys
import numpy as np
import torch
import cv2
import rasterio
from PIL import Image
from tqdm import tqdm

sys.path.append("/content/drive/MyDrive/MBRSC_Project/C2F-SemiCD")

folder_2019 = "./Processed Dataset Tiles/2019"
folder_2023 = "./Processed Dataset Tiles/2023"
output_folder = "./Model_Predictions_Tiles"
os.makedirs(output_folder, exist_ok=True)

checkpoint_path = (
    "in1__best_student_iou.pth"
)

THRESHOLD = 0.7
MIN_COMPONENT_AREA = 0  # Remove connected components smaller than this, not very significant since we already threshold

def load_tif_rgb(path):
    with rasterio.open(path) as src:
        img = src.read([1, 2, 3])          # CHW
        img = np.transpose(img, (1, 2, 0)) # HWC
    return img

def preprocess_image_hwc(img_hwc):
    img = img_hwc.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))    # CHW
    return torch.from_numpy(img).unsqueeze(0)  # 1,3,H,W

def predict_with_tta(model, A, B):
    # Test-time augmentation: average predictions across 4 orientations
    p1 = torch.sigmoid(model(A, B)[1])
    p2 = torch.sigmoid(model(A.flip(3), B.flip(3))[1]).flip(3)       # horizontal flip
    p3 = torch.sigmoid(model(A.flip(2), B.flip(2))[1]).flip(2)       # vertical flip
    p4 = torch.sigmoid(model(A.flip(2,3), B.flip(2,3))[1]).flip(2,3) # both flips
    return (p1 + p2 + p3 + p4) / 4.0

def remove_small_components(mask, min_area):
    # Remove connected components smaller than min_area pixels.
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for i in range(1, num_labels):  # skip background (label 0)
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == i] = 255
    return cleaned

def extract_id(filename):
    return os.path.splitext(os.path.basename(filename))[0]

def pad_to_multiple(img_hwc, multiple=32, pad_mode="reflect"):
    H, W, C = img_hwc.shape
    pad_h = (multiple - (H % multiple)) % multiple
    pad_w = (multiple - (W % multiple)) % multiple

    if pad_h == 0 and pad_w == 0:
        return img_hwc, (H, W)

    if pad_mode == "reflect":
        padded = np.pad(img_hwc, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    else:
        padded = np.pad(img_hwc, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant", constant_values=0)

    return padded, (H, W)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SemiModel().to(device)
model.eval()

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["best_student_net "])
print("Model loaded. TTA enabled, threshold:", THRESHOLD, "min component area:", MIN_COMPONENT_AREA)

# Match pairs
files_2023 = {}
for f in os.listdir(folder_2023):
    if f.lower().endswith(".tif"):
        tid = extract_id(f)
        if tid is not None:
            files_2023[tid] = f

pairs = []
for f19 in os.listdir(folder_2019):
    if not f19.lower().endswith(".tif"):
        continue
    tid = extract_id(f19)
    if tid is None:
        continue
    f23 = files_2023.get(tid, None)
    if f23 is not None:
        pairs.append((f19, f23, tid))

print("Matched pairs:", len(pairs))

# Inference
failed = []
t0_all = time.perf_counter()

for idx, (f19, f23, tid) in enumerate(tqdm(pairs, desc="Infer")):
    path_2019 = os.path.join(folder_2019, f19)
    path_2023 = os.path.join(folder_2023, f23)

    try:
        A_img = load_tif_rgb(path_2019)
        B_img = load_tif_rgb(path_2023)

        H = min(A_img.shape[0], B_img.shape[0])
        W = min(A_img.shape[1], B_img.shape[1])
        A_img = A_img[:H, :W, :]
        B_img = B_img[:H, :W, :]

        A_pad, (H0, W0) = pad_to_multiple(A_img, multiple=32, pad_mode="reflect")
        B_pad, _        = pad_to_multiple(B_img, multiple=32, pad_mode="reflect")

        A = preprocess_image_hwc(A_pad).to(device)
        B = preprocess_image_hwc(B_pad).to(device)

        with torch.no_grad():
            prob_map = predict_with_tta(model, A, B)[0, 0].cpu().numpy()

        # Crop back to original size
        prob_map = prob_map[:H0, :W0]

        # Threshold and clean
        mask = (prob_map >= THRESHOLD).astype(np.uint8) * 255
        mask = remove_small_components(mask, MIN_COMPONENT_AREA)

        out_png = os.path.join(output_folder, f"change_{tid}.png")
        Image.fromarray(mask).save(out_png)

        if idx < 3:
            print(f"[OK] {tid} saved -> {out_png} | orig={H0}x{W0}")

    except Exception as e:
        failed.append((tid, f19, f23, str(e)))
        print(f"[FAIL] {tid}: {e}")
        continue

t1_all = time.perf_counter()
print(f"\nDone. Processed {len(pairs) - len(failed)}/{len(pairs)} pairs in {(t1_all - t0_all)/60:.2f} minutes.")
print("Failed:", len(failed))

if failed:
    log_path = os.path.join(output_folder, "failed_pairs.txt")
    with open(log_path, "w") as f:
        for tid, f19, f23, err in failed:
            f.write(f"{tid}\t{f19}\t{f23}\t{err}\n")
    print("Failure log saved to:", log_path)

Georeference Model Prediction masks for visualization in ArcGIS

In [ ]:
import os
import rasterio
import numpy as np
from PIL import Image
from tqdm import tqdm

folder_2019 = "./Processed Dataset Tiles/2019"
png_folder = "./Model_Predictions_Tiles" # path to model predictions
geotiff_output_folder = "./Georeferenced Model Predictions"
os.makedirs(geotiff_output_folder, exist_ok=True)

def extract_id(filename: str) -> str:
    stem = os.path.splitext(os.path.basename(filename))[0]
    parts = stem.split("_")
    return parts[-1] if len(parts) > 1 else stem

tif_by_id = {}
for f in os.listdir(folder_2019):
    if f.lower().endswith(".tif"):
        tid = extract_id(f)
        tif_by_id[tid] = f

png_files = [f for f in os.listdir(png_folder) if f.lower().endswith(".png")]

for png_file in tqdm(png_files, desc="Georeferencing PNGs"):
    tid = extract_id(png_file)
    if not tid:
        continue

    ref_tif_name = tif_by_id.get(tid)
    if ref_tif_name is None:
        print(f"[WARN] No matching 2019 TIF found for PNG '{png_file}' (tid={tid})")
        continue

    ref_tif_path = os.path.join(folder_2019, ref_tif_name)
    png_path = os.path.join(png_folder, png_file)
    out_tif_path = os.path.join(geotiff_output_folder, f"change_{tid}.tif")

    # Read PNG as a single-band mask (uint8)
    mask = np.array(Image.open(png_path))
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    mask = mask.astype(np.uint8)

    # Copy georeferencing from reference tile
    with rasterio.open(ref_tif_path) as src:
        profile = src.profile.copy()

        # Ensure mask size matches the reference
        if (mask.shape[0] != src.height) or (mask.shape[1] != src.width):
            print(
                f"[WARN] Size mismatch for tid={tid}: "
                f"mask={mask.shape[1]}x{mask.shape[0]} vs ref={src.width}x{src.height}. "
                f"regenerate PNGs at reference size."
            )

        profile.update(
            dtype=rasterio.uint8,
            count=1,
            height=mask.shape[0],
            width=mask.shape[1],
            nodata=0,
            compress="lzw"
        )

        with rasterio.open(out_tif_path, "w", **profile) as dst:
            dst.write(mask, 1)

print("Finished restoring spatial information!")